<a href="https://colab.research.google.com/github/CharalampiaKal/cicids2017-progressive-feature-reduction/blob/main/notebooks/01_data_audit_and_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from pathlib import Path

# Create temporary dataset folders in the Colab runtime
DATA_DIR = Path("/content/data")
RAW_DIR = DATA_DIR / "raw"

DATA_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

# Official corrected CICIDS2017 dataset
DATASET_URL = (
    "https://intrusion-detection.distrinet-research.be/"
    "CNS2022/Datasets/CICIDS2017_improved.zip"
)
ZIP_PATH = DATA_DIR / "CICIDS2017_improved.zip"

# Download the dataset directly to Colab
!wget -c "$DATASET_URL" -O "$ZIP_PATH"

# Verify the download
print("\nDataset downloaded:", ZIP_PATH.exists())

if ZIP_PATH.exists():
    size_mb = ZIP_PATH.stat().st_size / (1024 ** 2)
    print(f"File size: {size_mb:.2f} MB")

--2026-08-23 18:16:30--  https://intrusion-detection.distrinet-research.be/CNS2022/Datasets/CICIDS2017_improved.zip
Resolving intrusion-detection.distrinet-research.be (intrusion-detection.distrinet-research.be)... 134.58.40.205
Connecting to intrusion-detection.distrinet-research.be (intrusion-detection.distrinet-research.be)|134.58.40.205|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.


Dataset downloaded: True
File size: 327.63 MB


In [4]:
import zipfile

# Extract the corrected dataset
with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
    zip_file.extractall(RAW_DIR)

# Find and display all extracted CSV files
csv_files = sorted(RAW_DIR.rglob("*.csv"))

print("CSV files extracted:", len(csv_files))

for file_path in csv_files:
    size_mb = file_path.stat().st_size / (1024 ** 2)
    print(f"{file_path.name}: {size_mb:.2f} MB")

CSV files extracted: 5
friday.csv: 271.98 MB
monday.csv: 198.25 MB
thursday.csv: 180.74 MB
tuesday.csv: 170.13 MB
wednesday.csv: 277.80 MB


In [5]:
import pandas as pd

# Read only the column names without loading the full dataset
schemas = {}

for file_path in csv_files:
    columns = pd.read_csv(file_path, nrows=0).columns.str.strip().tolist()
    schemas[file_path.name] = columns

# Use the first file as the reference schema
reference_file = csv_files[0].name
reference_columns = schemas[reference_file]

for filename, columns in schemas.items():
    print(
        f"{filename}: {len(columns)} columns | "
        f"Matches reference: {columns == reference_columns}"
    )

print("\nAll files have the same schema:",
      all(columns == reference_columns for columns in schemas.values()))

print("\nFirst 10 columns:")
print(reference_columns[:10])

print("\nLast 10 columns:")
print(reference_columns[-10:])

friday.csv: 91 columns | Matches reference: True
monday.csv: 91 columns | Matches reference: True
thursday.csv: 91 columns | Matches reference: True
tuesday.csv: 91 columns | Matches reference: True
wednesday.csv: 91 columns | Matches reference: True

All files have the same schema: True

First 10 columns:
['id', 'Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet']

Last 10 columns:
['Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min', 'ICMP Code', 'ICMP Type', 'Total TCP Flow Time', 'Label', 'Attempted Category']


In [7]:
from collections import Counter

file_audit = []
provided_label_counts = Counter()
modeling_label_counts = Counter()
attempted_category_counts = Counter()

# Process the dataset in chunks to avoid memory problems
for file_path in csv_files:
    file_rows = 0
    file_attempted = 0

    for chunk in pd.read_csv(
        file_path,
        usecols=["Label", "Attempted Category"],
        chunksize=200_000,
        low_memory=False
    ):
        labels = chunk["Label"].astype(str).str.strip()

        attempted_categories = pd.to_numeric(
            chunk["Attempted Category"],
            errors="coerce"
        ).fillna(-1).astype(int)

        is_attempted = attempted_categories.ne(-1)

        # Preserve the labels supplied in the dataset
        provided_label_counts.update(labels.value_counts().to_dict())

        # Following the dataset authors' recommendation,
        # attempted attacks will be treated as benign
        modeling_labels = labels.mask(is_attempted, "BENIGN")
        modeling_label_counts.update(modeling_labels.value_counts().to_dict())

        attempted_category_counts.update(
            attempted_categories[is_attempted].value_counts().to_dict()
        )

        file_rows += len(chunk)
        file_attempted += int(is_attempted.sum())

    file_audit.append({
        "File": file_path.name,
        "Rows": file_rows,
        "Attempted flows": file_attempted
    })

file_audit_df = pd.DataFrame(file_audit)

modeling_distribution_df = (
    pd.DataFrame(
        modeling_label_counts.items(),
        columns=["Label", "Count"]
    )
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)

print("Total rows:", file_audit_df["Rows"].sum())
print("Total attempted flows:", file_audit_df["Attempted flows"].sum())

print("\nRows and attempted flows per file:")
display(file_audit_df)

print("\nClass distribution after attempted flows are treated as benign:")
display(modeling_distribution_df)

Total rows: 2099976
Total attempted flows: 11979

Rows and attempted flows per file:


,File,Rows,Attempted flows
0,friday.csv,547557,4067
1,monday.csv,371624,0
2,thursday.csv,362076,1997
3,tuesday.csv,322078,39
4,wednesday.csv,496641,5876



Class distribution after attempted flows are treated as benign:


,Label,Count
0,BENIGN,1594545
1,Portscan,159066
2,DoS Hulk,158468
3,DDoS,95144
4,Infiltration - Portscan,71767
5,DoS GoldenEye,7567
6,FTP-Patator,3972
7,DoS Slowloris,3859
8,SSH-Patator,2961
9,DoS Slowhttptest,1740
